# AM01 Official Experiments

Notebook ufficiale Colab per il progetto AM01: dataset inspection, preprocessing, main protocol, ablation essenziali, diagnostica AAE e figure finali per il report.

Protocollo principale: `window_length=64`, `stride=16`, StandardScaler, MSE, seed 42. La sensitivity usa solo `w=32` e `w=64`; `w=128` e' escluso perche' riduce troppo il numero di finestre valutabili.

## 0. Parametri

In [ ]:
from pathlib import Path

REPO_URL = "https://github.com/Bernuz2003/AML_anomaliy_detection.git"  # opzionale: https://github.com/<user>/<repo>.git
PROJECT_DIR = Path('/content/am01-kuka-aae-anomaly-detection')
DRIVE_ROOT = Path('/content/drive/MyDrive/AM01')
DATA_DIR = DRIVE_ROOT / 'data' / 'KukaVelocityDataset'
OFFICIAL_ROOT = DRIVE_ROOT / 'results' / 'official'
CONFIG_DIR = OFFICIAL_ROOT / 'config'
TABLES_DIR = OFFICIAL_ROOT / 'tables'
FIGURES_DIR = OFFICIAL_ROOT / 'figures'
RUNS_DIR = OFFICIAL_ROOT / 'runs'
EXTENDED_DIR = OFFICIAL_ROOT / 'extended_scores'

RUN_PREPROCESSING = True
RUN_MAIN_EXPERIMENTS = True
RUN_CORE_ABLATIONS = True
RUN_AAE_DIAGNOSTICS = True
RUN_REPORT_FIGURES = True

PRIMARY_WINDOW_LENGTH = 64
PRIMARY_STRIDE = 16
SENSITIVITY_WINDOWS = [32, 64]
SEEDS_MAIN = [42]
SEEDS_STABILITY = [0, 1, 2]
SELECTION_METRIC = 'val_pr_auc'

print('DATA_DIR:', DATA_DIR)
print('OFFICIAL_ROOT:', OFFICIAL_ROOT)

## 1. Mount Drive e setup repo

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
import subprocess
import sys


def sh(cmd: str) -> None:
    print(f"\n$ {cmd}")
    subprocess.run(cmd, shell=True, check=True)

for path in [OFFICIAL_ROOT, CONFIG_DIR, TABLES_DIR, FIGURES_DIR, RUNS_DIR, EXTENDED_DIR]:
    path.mkdir(parents=True, exist_ok=True)

if not PROJECT_DIR.exists():
    if not REPO_URL:
        raise RuntimeError('PROJECT_DIR non esiste. Imposta REPO_URL o carica il repo in /content.')
    sh(f'git clone {REPO_URL} "{PROJECT_DIR}"')

os.chdir(PROJECT_DIR)
sys.path.insert(0, str(PROJECT_DIR / 'src'))
print('Working directory:', Path.cwd())

sh('pip install -q -r requirements.txt')
sh('python -m compileall -q src scripts')
sh(f'pip freeze > "{CONFIG_DIR / "environment.txt"}"')

## 2. Import e configurazione ufficiale

In [ ]:
import json
import shutil

import pandas as pd
import seaborn as sns
from IPython.display import Markdown, display

from am01.utils.config import load_config
import am01.reporting as rpt

sns.set_theme(style='whitegrid', context='notebook')

base_config = load_config('configs/ae_mlp.yaml')
base_config['windowing']['window_length'] = PRIMARY_WINDOW_LENGTH
base_config['windowing']['stride'] = PRIMARY_STRIDE
with (CONFIG_DIR / 'official_config.json').open('w', encoding='utf-8') as f:
    json.dump(base_config, f, indent=2)

print('Selection metric:', SELECTION_METRIC)
print('Sensitivity windows:', SENSITIVITY_WINDOWS)

## 3. Dataset inspection e figure iniziali

Genera le figure report-ready sul dataset, inclusa la sensitivity del numero di finestre per `w=32` e `w=64`.

In [ ]:
if RUN_REPORT_FIGURES:
    dataset_artifacts = rpt.dataset_report_artifacts(
        base_config,
        DATA_DIR,
        tables_dir=TABLES_DIR,
        figures_dir=FIGURES_DIR,
        window_lengths=SENSITIVITY_WINDOWS,
        primary_window_length=PRIMARY_WINDOW_LENGTH,
    )
    display(dataset_artifacts['dataset_composition'])
    display(dataset_artifacts['window_count_sensitivity'])
else:
    dataset_artifacts = {}

## 4. Preprocessing ufficiale

Data loading, split per run, scaling fit solo sui normali di training e windowing ufficiale.

In [ ]:
if RUN_PREPROCESSING:
    sh(f'python scripts/audit_data.py --config configs/ae_mlp.yaml --data "{DATA_DIR}" --output "{OFFICIAL_ROOT / "data_audit"}"')
    sh(f'python scripts/prepare_data.py --config configs/ae_mlp.yaml --data "{DATA_DIR}" --output "{OFFICIAL_ROOT / "preprocessed"}"')

preprocessing_summary = rpt.preprocessing_summary_table(
    OFFICIAL_ROOT / 'preprocessed',
    TABLES_DIR / 'preprocessing_summary.csv',
)
display(preprocessing_summary.style.format(precision=3))

## 5. Main protocol

Modelli principali: PCA, Isolation Forest, AE MLP, AAE MLP, AE Conv1D.

In [ ]:
main_configs = 'configs/pca.yaml configs/isolation_forest.yaml configs/ae_mlp.yaml configs/aae_mlp.yaml configs/ae_conv1d.yaml'
main_output = RUNS_DIR / 'main'
if RUN_MAIN_EXPERIMENTS:
    sh(
        f'python scripts/run_experiments.py '
        f'--configs {main_configs} '
        f'--data "{DATA_DIR}" '
        f'--output "{main_output}" '
        f'--seeds 42 '
        f'--skip-existing '
        f'--summary-name main_results_raw.csv'
    )

main_results = rpt.save_main_results(main_output, TABLES_DIR)
display(main_results.style.format(precision=4))

if RUN_REPORT_FIGURES:
    rpt.plot_main_result_figures(main_output, FIGURES_DIR)

## 6. AE vs AAE direct comparison

Questa e' la sezione direttamente collegata alla research question.

In [ ]:
ae_vs_aae = rpt.save_ae_vs_aae_comparison(main_results, TABLES_DIR, FIGURES_DIR)
display(ae_vs_aae.style.format(precision=4))

## 7. Ablation essenziali

Ablation incluse nel notebook ufficiale:

- window length: `32`, `64`;
- AAE: `latent_dim in {16, 32}`, `lambda_adv in {0.001, 0.01, 0.05, 0.1}`;
- preprocessing/loss: StandardScaler/RobustScaler e MSE/Huber.

In [ ]:
window_output = RUNS_DIR / 'window_sensitivity'
aae_ablation_output = RUNS_DIR / 'aae_ablation'
preprocessing_output = RUNS_DIR / 'preprocessing_loss'

if RUN_CORE_ABLATIONS:
    sh(
        f'python scripts/run_experiments.py '
        f'--configs configs/ae_mlp.yaml configs/aae_mlp.yaml '
        f'--data "{DATA_DIR}" '
        f'--output "{window_output}" '
        f'--seeds 42 '
        f'--window-lengths 32 64 '
        f'--skip-existing '
        f'--summary-name window_length_sensitivity_raw.csv'
    )
    sh(
        f'python scripts/run_experiments.py '
        f'--configs configs/aae_mlp.yaml '
        f'--data "{DATA_DIR}" '
        f'--output "{aae_ablation_output}" '
        f'--seeds 42 '
        f'--latent-dims 16 32 '
        f'--lambda-advs 0.001 0.01 0.05 0.1 '
        f'--skip-existing '
        f'--summary-name aae_ablation_raw.csv'
    )
    sh(
        f'python scripts/run_experiments.py '
        f'--configs configs/ae_mlp.yaml configs/aae_mlp.yaml '
        f'--data "{DATA_DIR}" '
        f'--output "{preprocessing_output}" '
        f'--seeds 42 '
        f'--scalers standard robust '
        f'--losses mse huber '
        f'--skip-existing '
        f'--summary-name preprocessing_loss_raw.csv'
    )

ablation_tables = rpt.save_ablation_tables_and_figures(
    window_runs_root=window_output,
    aae_runs_root=aae_ablation_output,
    preprocessing_runs_root=preprocessing_output,
    tables_dir=TABLES_DIR,
    figures_dir=FIGURES_DIR,
)
for name, table in ablation_tables.items():
    display(Markdown(f'### {name}'))
    display(table.head(30).style.format(precision=4))

## 8. AAE-specific scoring e latent diagnostics

Il run AAE viene selezionato usando `val_pr_auc`, non il test. Qui verifichiamo se latent space e discriminator contengono segnale utile oltre alla reconstruction error.

In [ ]:
def collect_existing_metrics(roots):
    frames = []
    for root in roots:
        root = Path(root)
        if root.exists():
            try:
                frames.append(rpt.collect_run_metrics(root))
            except FileNotFoundError:
                pass
    if not frames:
        raise FileNotFoundError('Nessun run disponibile per la selezione AE/AAE.')
    return pd.concat(frames, ignore_index=True)

candidate_runs = collect_existing_metrics([main_output, aae_ablation_output, preprocessing_output])
candidate_runs.to_csv(TABLES_DIR / 'candidate_ae_aae_runs.csv', index=False)

ae_candidates = candidate_runs[candidate_runs['model_key'] == 'ae_mlp'].dropna(subset=[SELECTION_METRIC])
aae_candidates = candidate_runs[candidate_runs['model_key'] == 'aae_mlp'].dropna(subset=[SELECTION_METRIC])
BEST_AE_RUN = Path(ae_candidates.sort_values(SELECTION_METRIC, ascending=False).iloc[0]['run_dir'])
BEST_AAE_RUN = Path(aae_candidates.sort_values(SELECTION_METRIC, ascending=False).iloc[0]['run_dir'])

display(Markdown(f'**Selected AE:** `{BEST_AE_RUN}`  \n**Selected AAE:** `{BEST_AAE_RUN}`'))

if RUN_AAE_DIAGNOSTICS:
    diag = rpt.aae_diagnostics_artifacts(
        ae_run_dir=BEST_AE_RUN,
        aae_run_dir=BEST_AAE_RUN,
        tables_dir=TABLES_DIR,
        figures_dir=FIGURES_DIR,
        extended_dir=EXTENDED_DIR,
        selection_metric=SELECTION_METRIC,
    )
    display(diag['score_table'].style.format(precision=4))
    if not diag['per_feature'].empty:
        display(diag['per_feature'].sort_values('delta').head(20).style.format(precision=5))
    if not diag['per_action'].empty:
        display(diag['per_action'].head(30).style.format(precision=4))
else:
    diag = {'score_table': pd.read_csv(TABLES_DIR / 'aae_specific_scores.csv')}

## 9. Summary finale e manifest artefatti

In [ ]:
aae_specific = pd.read_csv(TABLES_DIR / 'aae_specific_scores.csv') if (TABLES_DIR / 'aae_specific_scores.csv').exists() else None
window_sensitivity = pd.read_csv(TABLES_DIR / 'window_length_sensitivity.csv') if (TABLES_DIR / 'window_length_sensitivity.csv').exists() else None
rpt.write_official_summary(
    output_path=OFFICIAL_ROOT / 'summary.md',
    main_results=main_results,
    ae_vs_aae=ae_vs_aae,
    aae_specific=aae_specific,
    window_sensitivity=window_sensitivity,
)
display(Markdown((OFFICIAL_ROOT / 'summary.md').read_text(encoding='utf-8')))

artifacts = sorted([p for p in OFFICIAL_ROOT.rglob('*') if p.is_file()])
manifest = pd.DataFrame({'artifact': [str(p.relative_to(OFFICIAL_ROOT)) for p in artifacts]})
manifest.to_csv(TABLES_DIR / 'artifact_manifest.csv', index=False)
display(manifest)